In [4]:
def is_valid_sequence(expr: str)->bool:
    if len(list(expr))%2 == 0:
        check_tup = [('{','}'),('[',']'),('(',')'),('<','>')]
        first_half = list(expr)[:round(len(expr)/2)]        
        second_half_revr = list(reversed(list(expr)[round(len(expr)/2):]))
        for count in range(0, round(len(list(expr))/2)):
            if (first_half[count],second_half_revr[count]) in check_tup:
                continue
            else:
                return False
    else:
        return False
    return True                                                     
     

In [5]:
print(is_valid_sequence("{([<>])}"))
print(is_valid_sequence("{([<>)]}"))
print(is_valid_sequence("{([<>])"))
print(is_valid_sequence("{([<>])}}}"))

True
False
False
False


In [6]:
def check_and_convert(value: str):
    try:
        result = int(value)
        print(f"Success: {value} converted to int -> {result}")
        return result
    except ValueError:
        print(f"Failed: '{value}' cannot be directly converted to an int.")
        return None

def flatten_list(expr: list)->list:
    final_list: list = []
    for count in expr:
        if isinstance(count, int):
            final_list.append(count)
        elif isinstance(count, str) and isinstance(check_and_convert(count), int):
            final_list.append(int(count))
        elif isinstance(count, list):
            final_list.extend(flatten_list(count))
                
    return final_list
            

In [7]:
print(flatten_list([1,[2,"3",["four",5]],"6",7]))

Success: 3 converted to int -> 3
Failed: 'four' cannot be directly converted to an int.
Success: 6 converted to int -> 6
[1, 2, 3, 5, 6, 7]


In [8]:
from functools import wraps
from collections import OrderedDict

def custom_lru_cache(maxsize=128):
    def decorator(func):
        # Store cache inside the closure
        cache = OrderedDict()

        @wraps(func)
        def wrapper(*args, **kwargs):
            # Create a unique, hashable cache key from positional and keyword arguments
            key = (args, tuple(sorted(kwargs.items())))

            # Cache Hit
            if key in cache:
                cache.move_to_end(key)  # Mark as most recently used
                return cache[key]

            # Cache Miss: Compute the result
            result = func(*args, **kwargs)
            cache[key] = result

            # Check capacity and evict the oldest item if full
            if len(cache) > maxsize:
                cache.popitem(last=False)  # popitem(last=False) removes the oldest item (FIFO)

            return result
            
        # Optional: Add utility methods to inspect or clear the cache
        def cache_clear():
            cache.clear()
            
        wrapper.cache_clear = cache_clear
        return wrapper
        
    return decorator


In [9]:
import time

@custom_lru_cache(maxsize=3)
def heavy_calculation(n):
    print(f"Computing {n}...")
    time.sleep(0.5)  # Simulate slow work
    return n * 2

# 1. Populate the cache
print(heavy_calculation(1))  # Computes (Cache: [1])
print(heavy_calculation(2))  # Computes (Cache: [1, 2])
print(heavy_calculation(3))  # Computes (Cache: [1, 2, 3])

# 2. Cache Hit (Moves '2' to the most recently used position)
print(heavy_calculation(2))  # Instant result! (Cache: [1, 3, 2])

# 3. Cache Eviction (Triggers because maxsize is 3)
print(heavy_calculation(4))  # Computes. '1' is evicted because it's the oldest! (Cache: [3, 2, 4])

# 4. Verifying eviction
print(heavy_calculation(1))  # Computes again because it was evicted.


Computing 1...
2
Computing 2...
4
Computing 3...
6
4
Computing 4...
8
Computing 1...
2


In [10]:
def rle_encode(text: str) -> str:
    """Encodes a string using Run-Length Encoding (e.g., 'AAAABBBCC' -> '4A3B2C')."""
    if not text:
        return ""

    encoded = []
    current_char = text[0]
    count = 1

    # Iterate through the string starting from the second character
    for char in text[1:]:
        if char == current_char:
            count += 1
        else:
            encoded.append(f"{count}{current_char}")
            current_char = char
            count = 1
            
    # Append the final group
    encoded.append(f"{count}{current_char}")
    return "".join(encoded)


def rle_decode(text: str) -> str:
    """Decodes a Run-Length Encoded string back to its original form."""
    if not text:
        return ""

    decoded = []
    count_str = ""

    for char in text:
        if char.isdigit():
            count_str += char  # Build multi-digit counts (e.g., "12")
        else:
            # If no digit preceded the character, assume a count of 1
            count = int(count_str) if count_str else 1
            decoded.append(char * count)
            count_str = ""  # Reset count for the next character

    return "".join(decoded)


In [11]:
# 1. Encoding
original_text = "AAAAABBBCCDAA"
encoded_text = rle_encode(original_text)

print(f"Original: {original_text}")
print(f"Encoded:  {encoded_text}")  # Output: 5A3B2C1D2A

# 2. Decoding
decoded_text = rle_decode(encoded_text)
print(f"Decoded:  {decoded_text}")  # Output: AAAAABBBCCDAA

# 3. Verifying integrity
assert original_text == decoded_text


Original: AAAAABBBCCDAA
Encoded:  5A3B2C1D2A
Decoded:  AAAAABBBCCDAA


In [12]:
def highest_alphabet(expr: str) -> str:
    hi_alpha: dict = {}
    for val in list(expr):
        if val not in hi_alpha:
            hi_alpha[val] = 1
        else:
            hi_alpha[val] +=1
    return hi_alpha

my_dict = highest_alphabet("monsoon")
highest_key = max(my_dict, key=my_dict.get)
print(highest_key)  # Output: o 

o


In [13]:
from typing import Callable

def wrapit_string(func: Callable)->Callable:
    def inside_wrap(str1:str, str2:str) -> str:
        print("Before decorator {0}".format(func(str1, str2)))
        print("After decorator {0}".format(str1.upper()+" "+str2.upper()))
        return "decortor called successfully"
    return inside_wrap

@wrapit_string
def lower_string(str1: str, str2: str):
    return str1.lower() + " " + str2.lower()

print(lower_string("vinod", "vukkalam"))

Before decorator vinod vukkalam
After decorator VINOD VUKKALAM
decortor called successfully
